# Linear Systems and Vector Spaces

Notebook for **Chapter 7** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Systems of Equations as Ax = b: Two Pictures

In [ ]:
import numpy as np

A = np.array([[2, 1], [1, 3]])
b = np.array([7, 11])
x = np.linalg.solve(A, b)           # solve Ax = b
print(np.round(x, 6))
print(x[0] * A[:, 0] + x[1] * A[:, 1])   # the same b, built from columns
print(np.allclose(A @ x, b))

[2. 3.]
[ 7. 11.]
True


## 7.2 Gaussian Elimination, Echelon Forms and the Three Outcomes

In [ ]:
import numpy as np

def rref(M, tol=1e-12):
    M = M.astype(float).copy()
    rows, cols = M.shape
    r, pivots = 0, []
    for c in range(cols):
        if r == rows:
            break
        p = r + np.argmax(np.abs(M[r:, c]))     # best pivot in this column
        if abs(M[p, c]) < tol:
            continue                              # free column
        M[[r, p]] = M[[p, r]]                     # swap rows
        M[r] = M[r] / M[r, c]                     # make the pivot 1
        for i in range(rows):
            if i != r:
                M[i] = M[i] - M[i, c] * M[r]      # clear the column
        pivots.append(c)
        r += 1
    return M, pivots

A = np.array([[1, 1, 1], [1, 2, 3], [2, 3, 4]])
for rhs in ([3, 6, 10], [3, 6, 9]):
    R, piv = rref(np.column_stack([A, rhs]))
    print("rhs", rhs, "pivot columns", piv)
    print(np.round(R, 6) + 0.0)

B = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])
R, piv = rref(np.column_stack([B, [8, -11, -3]]))
print("unique:", R[:, -1], piv)

rhs [3, 6, 10] pivot columns [0, 1, 3]
[[ 1.  0. -1.  0.]
 [ 0.  1.  2.  0.]
 [ 0.  0.  0.  1.]]
rhs [3, 6, 9] pivot columns [0, 1]
[[ 1.  0. -1.  0.]
 [ 0.  1.  2.  3.]
 [ 0.  0.  0.  0.]]
unique: [ 2.  3. -1.] [0, 1, 2]


## 7.3 Rank, Free Variables, Homogeneous Systems and Rank-Nullity

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
a, b, c = rng.normal(size=(3, 5))        # three random columns of length 5
d = a + 2 * c                            # redundant feature
X = np.column_stack([a, b, c, d])
print("shape", X.shape, "rank", np.linalg.matrix_rank(X))
x0 = np.array([1, 0, 2, -1])
print("X @ x0 is zero:", np.allclose(X @ x0, 0))

shape (5, 4) rank 3
X @ x0 is zero: True


## 7.4 Vector Spaces, Subspaces and the Four Fundamental Subspaces

In [ ]:
import numpy as np
from scipy.linalg import null_space, orth

A = np.array([[1, 2, 1, 1], [2, 4, 3, 4], [1, 2, 0, -1]], dtype=float)
r = np.linalg.matrix_rank(A)
N = null_space(A)          # basis of N(A):    columns, shape (4, n - r)
L = null_space(A.T)        # basis of N(A^T):  columns, shape (3, m - r)
C = orth(A)                # basis of C(A):    columns, shape (3, r)
print("rank", r, "| dim null", N.shape[1], "| dim left null", L.shape[1], "| dim col", C.shape[1])

print("A @ N = 0:", np.allclose(A @ N, 0))
print("hand null vectors work:", np.allclose(A @ np.array([[-2, 1, 0, 0], [1, 0, -2, 1]]).T, 0))
y = np.array([-3, 1, 1])
print("hand left-null vector works:", np.allclose(y @ A, 0))

# is b in the column space?  yes if appending b does not raise the rank
for b in ([5, 13, 2], [1, 0, 0]):
    rank_aug = np.linalg.matrix_rank(np.column_stack([A, b]))
    print("b =", b, "in column space:", rank_aug == r, "| y . b =", y @ np.array(b))

rank 2 | dim null 2 | dim left null 1 | dim col 2
A @ N = 0: True
hand null vectors work: True
hand left-null vector works: True
b = [5, 13, 2] in column space: True | y . b = 0
b = [1, 0, 0] in column space: False | y . b = -3


In [ ]:
import numpy as np
from scipy.linalg import null_space

weekday = np.array([1, 1, 1, 0, 1, 0, 1], dtype=float)
X = np.column_stack([weekday, 1 - weekday, np.ones(7)])
print("rank", np.linalg.matrix_rank(X), "of", X.shape[1], "columns")
v = null_space(X)[:, 0]
print("null direction (scaled):", np.round(np.abs(v) / np.abs(v).max(), 3))

rank 2 of 3 columns
null direction (scaled): [1. 1. 1.]


## 7.5 LU Decomposition: Elimination Remembered

In [ ]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve

def lu_nopivot(A):
    n = len(A)
    L, U = np.eye(n), A.astype(float).copy()
    for j in range(n - 1):
        for i in range(j + 1, n):
            L[i, j] = U[i, j] / U[j, j]       # the multiplier goes into L
            U[i] = U[i] - L[i, j] * U[j]      # the row operation
    return L, U

A = np.array([[2, 1, 1], [4, -6, 0], [-2, 7, 2]])
L, U = lu_nopivot(A)
print(L + 0.0)
print(U + 0.0)
print("L @ U equals A:", np.allclose(L @ U, A))

lu, piv = lu_factor(A.astype(float))            # factor ONCE (with pivoting)
B = np.array([[5, 1], [-2, 0], [9, 0]], dtype=float)   # two right-hand sides as columns
X = lu_solve((lu, piv), B)                      # cheap solves
print(np.round(X, 4))
print("det from U:", round(float(np.prod(np.diag(U))), 6), "| np.linalg.det:", round(float(np.linalg.det(A)), 6))

[[ 1.  0.  0.]
 [ 2.  1.  0.]
 [-1. -1.  1.]]
[[ 2.  1.  1.]
 [ 0. -8. -2.]
 [ 0.  0.  1.]]
L @ U equals A: True
[[ 1.    0.75]
 [ 1.    0.5 ]
 [ 2.   -1.  ]]
det from U: -16.0 | np.linalg.det: -16.0


In [ ]:
import numpy as np

eps = 1e-20
A = np.array([[eps, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])

# elimination WITHOUT swapping rows: pivot is the tiny eps
m = A[1, 0] / A[0, 0]                  # multiplier = 1e20
a22 = A[1, 1] - m * A[0, 1]            # 1 - 1e20, rounds to -1e20
b2 = b[1] - m * b[0]                   # 2 - 1e20, rounds to -1e20
y = b2 / a22
x = (b[0] - A[0, 1] * y) / A[0, 0]
print("no pivoting:  x =", x, " y =", y)
print("with pivoting (np.linalg.solve):", np.linalg.solve(A, b))

no pivoting:  x = 0.0  y = 1.0
with pivoting (np.linalg.solve): [1. 1.]


## 7.6 Overdetermined and Underdetermined Systems: Least Squares and Minimum Norm

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4.0])
y = np.array([1, 3, 4, 4.0])
A = np.column_stack([np.ones(4), x])          # columns: ones (intercept), x
xhat = np.linalg.solve(A.T @ A, A.T @ y)      # normal equations
print("c, m =", xhat)
print("lstsq  =", np.linalg.lstsq(A, y, rcond=None)[0])
print("polyfit (m, c) =", np.polyfit(x, y, 1))
p = A @ xhat
e = y - p
print("fitted", p, "errors", e)
print("A.T @ e =", np.round(A.T @ e, 12) + 0.0)
print("R^2 =", round(1 - e @ e / ((y - y.mean()) @ (y - y.mean())), 4))

c, m = [0.5 1. ]
lstsq  = [0.5 1. ]
polyfit (m, c) = [1.  0.5]
fitted [1.5 2.5 3.5 4.5] errors [-0.5  0.5  0.5 -0.5]
A.T @ e = [0. 0.]
R^2 = 0.8333


In [ ]:
import numpy as np

# overdetermined: the 3 x 2 system from Problem 7.6.2
A = np.array([[1, 0], [0, 1], [1, 1.0]])
b = np.array([1, 1, 0.0])
print("pinv solution :", np.round(np.linalg.pinv(A) @ b, 4))

# underdetermined: Problem 7.6.3
A2 = np.array([[1, 1, 0], [0, 1, 1.0]])
b2 = np.array([2, 3.0])
x = np.linalg.pinv(A2) @ b2
print("min-norm      :", np.round(x, 4), " 5/3 =", round(5 / 3, 4))
print("same formula  :", np.round(A2.T @ np.linalg.solve(A2 @ A2.T, b2), 4))
print("lstsq agrees  :", np.allclose(np.linalg.lstsq(A2, b2, rcond=None)[0], x))
print("exact?        :", np.allclose(A2 @ x, b2), " length^2 =", round(float(x @ x), 4))

pinv solution : [0.3333 0.3333]
min-norm      : [0.3333 1.6667 1.3333]  5/3 = 1.6667
same formula  : [0.3333 1.6667 1.3333]
lstsq agrees  : True
exact?        : True  length^2 = 4.6667


## 7.7 Data-Science Applications: Five Jobs for Ax = b

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 200
x1, x2 = rng.normal(size=n), rng.normal(size=n)
y = 3 + 2 * x1 - 1.5 * x2 + rng.normal(scale=0.5, size=n)
X = np.column_stack([np.ones(n), x1, x2])

beta_normal = np.linalg.solve(X.T @ X, X.T @ y)
beta_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]
print("normal equations:", np.round(beta_normal, 3))
print("lstsq           :", np.round(beta_lstsq, 3))
resid = y - X @ beta_lstsq
print("X.T @ resid ~ 0 :", np.allclose(X.T @ resid, 0, atol=1e-8))
print("residual sum    :", round(float(resid.sum()), 10) + 0.0)

normal equations: [ 3.005  2.034 -1.47 ]
lstsq           : [ 3.005  2.034 -1.47 ]
X.T @ resid ~ 0 : True
residual sum    : 0.0


In [ ]:
import numpy as np
from scipy.linalg import null_space

A = np.array([[1, 1, 0, 0, 0],
              [1, 0, -1, -1, 0],
              [0, 1, 1, 0, -1],
              [0, 0, 0, 1, 1]], dtype=float)
b = np.array([100, 0, 0, 100], dtype=float)
print("rank", np.linalg.matrix_rank(A), "nullity", null_space(A).shape[1])
print("consistent:", np.linalg.matrix_rank(np.column_stack([A, b])) == np.linalg.matrix_rank(A))
x = np.array([60, 40, 10, 50, 50.0])             # a = 60, b = 10
print("a=60, b=10 balances:", np.allclose(A @ x, b))
print("min-norm flow:", np.round(np.linalg.lstsq(A, b, rcond=None)[0], 2) + 0.0)

rank 3 nullity 2
consistent: True
a=60, b=10 balances: True
min-norm flow: [50. 50.  0. 50. 50.]


In [ ]:
import numpy as np

A = np.array([[1, 1, 1], [4, 1, 2], [2, 1, 4]], dtype=float)   # weight, protein/10, fat/5
b = np.array([100, 250, 230], dtype=float)
print("kg of soy, corn, oats:", np.linalg.solve(A, b))
print("det A =", round(float(np.linalg.det(A)), 6), "(nonzero, so the answer is unique)")

kg of soy, corn, oats: [40. 30. 30.]
det A = -8.0 (nonzero, so the answer is unique)


In [ ]:
import numpy as np

A = np.array([[2, 1, 1], [1, 3, 1], [1, 1, 2]], dtype=float)
b = np.array([9, 11.5, 11.5])
print("prices:", np.linalg.solve(A, b))

A4 = np.array([[2, 1, 1], [1, 3, 1], [3, 4, 2]], dtype=float)   # row 3 = row 1 + row 2
b4 = np.array([9, 11.5, 20.5])
print("rank:", np.linalg.matrix_rank(A4))
for t in (0.0, 0.1):
    prices = np.array([1.5, 2, 4]) + t * np.array([2, 1, -5])
    print("t =", t, "prices", prices, "totals", A4 @ prices)

prices: [1.5 2.  4. ]
rank: 2
t = 0.0 prices [1.5 2.  4. ] totals [ 9.  11.5 20.5]
t = 0.1 prices [1.7 2.1 3.5] totals [ 9.  11.5 20.5]


In [ ]:
import numpy as np

def fit(seed):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=100)
    x2 = x1 + rng.normal(scale=1e-3, size=100)      # nearly a copy of x1
    y = 3 * x1 + rng.normal(scale=0.1, size=100)
    X = np.column_stack([x1, x2])
    return X, np.linalg.lstsq(X, y, rcond=None)[0]

for seed in (1, 2, 3):
    X, beta = fit(seed)
    print("seed", seed, "coefficients", np.round(beta, 1), "sum", round(float(beta.sum()), 2))
print("log10 cond(X)      =", round(float(np.log10(np.linalg.cond(X))), 1))
print("log10 cond(X^T X)  =", round(float(np.log10(np.linalg.cond(X.T @ X))), 1))

seed 1 coefficients [ 3.2 -0.2] sum 3.0
seed 2 coefficients [11. -8.] sum 3.02
seed 3 coefficients [-13.9  16.9] sum 3.01
log10 cond(X)      = 3.3
log10 cond(X^T X)  = 6.7


## 7.8 Numerical Health and the Expert Layer

In [ ]:
import numpy as np
from scipy.linalg import hilbert

A = np.array([[1, 1], [1, 1.0001]])
print("cond of the 2x2 example: %.1e" % np.linalg.cond(A))
print("x for b=(2, 2.0001):", np.round(np.linalg.solve(A, [2, 2.0001]), 6))
print("x for b=(2, 2.0002):", np.round(np.linalg.solve(A, [2, 2.0002]), 6))

for n in (4, 8, 12):
    H = hilbert(n)
    x_true = np.ones(n)
    x_hat = np.linalg.solve(H, H @ x_true)
    rel_err = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
    print("n =", n, " cond = %.1e" % np.linalg.cond(H), " relative error of solve: %.0e" % rel_err)

cond of the 2x2 example: 4.0e+04
x for b=(2, 2.0001): [1. 1.]
x for b=(2, 2.0002): [0. 2.]
n = 4  cond = 1.6e+04  relative error of solve: 3e-14
n = 8  cond = 1.5e+10  relative error of solve: 8e-08
n = 12  cond = 1.7e+16  relative error of solve: 1e-01


In [ ]:
import numpy as np
from scipy.linalg import null_space

A = np.array([[1, 2, 1, 1], [2, 4, 3, 4], [1, 2, 0, -1]], dtype=float)
x = np.array([1, 1, 1, 1.0])                  # one solution of A x = (5, 13, 2)
P_row = np.linalg.pinv(A) @ A                 # projection onto the row space
x_row = P_row @ x
x_null = x - x_row
print("row part :", np.round(x_row, 4))
print("null part:", np.round(x_null, 4))
print("same b   :", np.allclose(A @ x_row, A @ x))
print("null part is in N(A):", np.allclose(A @ x_null, 0))
print("perpendicular:", np.isclose(x_row @ x_null, 0))
print("lengths: full %.4f, row part %.4f" % (np.linalg.norm(x), np.linalg.norm(x_row)))

row part : [0.6154 1.2308 0.8462 1.0769]
null part: [ 0.3846 -0.2308  0.1538 -0.0769]
same b   : True
null part is in N(A): True
perpendicular: True
lengths: full 2.0000, row part 1.9415


In [ ]:
import numpy as np

A = np.column_stack([np.ones(4), [1, 2, 3, 4.0]])
P = A @ np.linalg.inv(A.T @ A) @ A.T
print("P @ P equals P :", np.allclose(P @ P, P))
print("P symmetric    :", np.allclose(P, P.T))
print("leverages      :", np.round(np.diag(P), 4), " trace =", round(float(np.trace(P)), 4))
y = np.array([1, 3, 4, 4.0])
print("hat y          :", np.round(P @ y, 4))
print("residual       :", np.round((np.eye(4) - P) @ y, 4))

P @ P equals P : True
P symmetric    : True
leverages      : [0.7 0.3 0.3 0.7]  trace = 2.0
hat y          : [1.5 2.5 3.5 4.5]
residual       : [-0.5  0.5  0.5 -0.5]


In [ ]:
import numpy as np
from scipy.linalg import solve_triangular

A = np.column_stack([np.ones(4), [1, 2, 3, 4.0]])
y = np.array([1, 3, 4, 4.0])
Q, R = np.linalg.qr(A)                       # reduced QR: Q is 4x2, R is 2x2
x_qr = solve_triangular(R, Q.T @ y)          # back substitution on R x = Q^T y
print("QR solution   :", np.round(x_qr, 4))
print("Q^T Q = I     :", np.allclose(Q.T @ Q, np.eye(2)))
print("cond(A) = %.3f, cond(A^T A) = %.3f, cond(A)^2 = %.3f" % (
    np.linalg.cond(A), np.linalg.cond(A.T @ A), np.linalg.cond(A) ** 2))

QR solution   : [0.5 1. ]
Q^T Q = I     : True
cond(A) = 7.469, cond(A^T A) = 55.782, cond(A)^2 = 55.782


In [ ]:
import numpy as np

A = np.array([[4, 1, 0], [1, 5, 1], [0, 1, 3.0]])
b = np.array([5, 7, 4.0])
x_true = np.ones(3)

D = np.diag(A)
R = A - np.diag(D)
x = np.zeros(3)
for k in range(1, 16):
    x = (b - R @ x) / D                       # Jacobi step
    if k in (1, 2, 15):
        print("Jacobi step", k, np.round(x, 4), "error %.1e" % np.linalg.norm(x - x_true))

x = np.zeros(3)
alpha = 1 / np.linalg.eigvalsh(A).max()       # safe step size
for k in range(1, 16):
    x = x - alpha * (A @ x - b)               # gradient step on 0.5 x^T A x - b^T x
print("Gradient step 15", np.round(x, 4), "error %.1e" % np.linalg.norm(x - x_true))

Jacobi step 1 [1.25   1.4    1.3333] error 5.8e-01
Jacobi step 2 [0.9    0.8833 0.8667] error 2.0e-01
Jacobi step 15 [1. 1. 1.] error 1.7e-07
Gradient step 15 [0.9999 1.0001 0.9998] error 2.0e-04


## Chapter recap

## Practice set

Problem 7.9.1 Write 3x + 2y = 12, x − y = 1 as Ax = b and solve by elimination.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch07.html